# DSC 411/511
# Association Analysis on Ames




---
# Section 0: Setup

In [ ]:
!pip install mlxtend --quiet

In [ ]:
import os, warnings, logging
os.environ['PYTHONWARNINGS'] = 'ignore'
warnings.simplefilter('ignore')
warnings.filterwarnings('ignore', category=DeprecationWarning)
logging.getLogger('py.warnings').setLevel(logging.ERROR)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings('ignore')

from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

pd.set_option('display.width', 200)
print('ready')

In [ ]:
DATA_URL = 'https://jse.amstat.org/v19n3/decock/AmesHousing.txt'
df = pd.read_csv(DATA_URL, sep='\t')       # tab separated
print('Rows:', df.shape[0], ' Columns:', df.shape[1])

---
# Section 1: Houses as Baskets

A supermarket basket is a list of products. A house is a list of facts about itself.

> {large house, high quality, big garage, built after 1990}

Every fact must be **true or false**. Association analysis cannot read a number.

In [ ]:
B = pd.DataFrame(index=df.index)
B['Large house']       = df['Gr Liv Area'] > df['Gr Liv Area'].median()
B['Large lot']         = df['Lot Area'] > df['Lot Area'].median()
B['High quality']      = df['Overall Qual'] >= 7
B['Built after 1990']  = df['Year Built'] >= 1990
B['Big garage']        = df['Garage Area'].fillna(0) > 500
B['Finished basement'] = df['Total Bsmt SF'].fillna(0) > 800
B['2+ bathrooms']      = df['Full Bath'] >= 2
B['Many rooms']        = df['TotRms AbvGrd'] >= 7
B['Has fireplace']     = df['Fireplaces'] > 0

# SalePrice is deliberately NOT a fact in the basket.
# Put the answer in and the rules would just rediscover it.
# We want to know which FEATURES co-occur, not how to predict price.

print(f'{len(B):,} houses, {B.shape[1]} facts')
print()
print('HOW COMMON IS EACH FACT?')
print((B.mean() * 100).round(1).sort_values(ascending=False).to_string())
print()
print(f'Average facts per house: {B.sum(axis=1).mean():.1f}')

In [ ]:
print('THE FIRST FIVE HOUSES AS BASKETS')
print()
for i in B.index[:5]:
    facts = [c for c in B.columns if B.loc[i, c]]
    print(f'   house {i}:  {{{", ".join(facts)}}}')

### A warning before we mine anything

**A fact true for nearly every house is useless.** It will appear in almost every rule
and its lift will be close to 1 by definition.

**A fact true for almost nobody is also useless.** Nothing will reach the support
threshold.

In [ ]:
too_common = B.columns[B.mean() > 0.95].tolist()
too_rare   = B.columns[B.mean() < 0.05].tolist()

print(f'True for over 95% of houses : {too_common if too_common else "none"}')
print(f'True for under 5% of houses : {too_rare if too_rare else "none"}')

if too_common or too_rare:
    B = B.drop(columns=too_common + too_rare)
    print(f'Dropped. {B.shape[1]} facts remain.')
else:
    print('Nothing to drop.')

---
# Section 2: Frequent Itemsets

A **frequent itemset** is any combination of facts that appears often enough to pass
your minimum support threshold.

**Support** is the fraction of houses containing that combination.

- **C1:** count the support of every single fact
- **L1:** keep the ones that pass minimum support
- **Ck:** join L(k-1) with itself, prune any candidate with an infrequent subset
- **Lk:** scan, count, keep the ones that pass
- Repeat until Lk is empty

In [ ]:
MIN_SUPPORT = 0.10          # a combination must appear in 10% of houses

fi = apriori(B, min_support=MIN_SUPPORT, use_colnames=True)
fi['size'] = fi.itemsets.apply(len)

print(f'Minimum support: {MIN_SUPPORT:.0%}  '
      f'({int(MIN_SUPPORT*len(B)):,} of {len(B):,} houses)')
print()
print('FREQUENT ITEMSETS BY SIZE')
print(fi.groupby('size').size().rename('count').to_string())
print()
print(f'Total: {len(fi):,}')

In [ ]:
print('L1: the single facts that passed')
print()
for _, r in fi[fi['size'] == 1].sort_values('support', ascending=False).iterrows():
    print(f'   {list(r.itemsets)[0]:<22}{r.support:.3f}')
print()
print('L2: the ten most common PAIRS')
print()
for _, r in fi[fi['size'] == 2].sort_values('support', ascending=False).head(10).iterrows():
    print(f'   {" + ".join(sorted(r.itemsets)):<48}{r.support:.3f}')

In [ ]:
print('THE LARGEST COMBINATIONS THAT STILL PASS')
print()
biggest = fi['size'].max()
for _, r in fi[fi['size'] == biggest].sort_values('support', ascending=False).head(5).iterrows():
    print(f'   support {r.support:.3f}   {{{", ".join(sorted(r.itemsets))}}}')
print()
print(f'No combination of {biggest+1} facts reaches {MIN_SUPPORT:.0%}. That is where Apriori stopped.')

---
# Section 3: From Itemsets to Rules

A frequent itemset is not yet useful. Split it into an **if** and a **then**.

```
{large house, high quality}  ->  {big garage}

support    = fraction of houses with ALL of it
confidence = of the houses with the left side, what fraction have the right
lift       = confidence divided by how common the right side is anyway
```

In [ ]:
rules = association_rules(fi, metric='lift', min_threshold=1.0)
rules = rules[rules.consequents.apply(len) == 1]          # one fact on the right
rules['IF']   = rules.antecedents.apply(lambda s: ' AND '.join(sorted(s)))
rules['THEN'] = rules.consequents.apply(lambda s: list(s)[0])

print(f'{len(rules):,} rules with a single fact on the right')
print()
print('ONE RULE, BROKEN DOWN')
r = rules.sort_values('lift', ascending=False).iloc[0]
print()
print(f'   IF    {r.IF}')
print(f'   THEN  {r.THEN}')
print()
print(f'   support    {r.support:.3f}   '
      f'{int(r.support*len(B)):,} houses have all of it')
print(f'   confidence {r.confidence:.3f}   of the houses with the IF, '
      f'{r.confidence*100:.0f}% also have the THEN')
print(f'   base rate  {r["consequent support"]:.3f}   '
      f'{r["consequent support"]*100:.0f}% of ALL houses have the THEN')
print(f'   lift       {r.lift:.3f}   = {r.confidence:.3f} / {r["consequent support"]:.3f}')
print()
print(f'   So the IF makes the THEN {r.lift:.2f} times more likely than the base rate.')

### Why confidence alone will fool you

| Lift | Meaning |
|---|---|
| **above 1** | They attract. A real association |
| **exactly 1** | Independent. Confidence is misleading you |
| **below 1** | They repel. One makes the other less likely |

A fact present in 80 percent of houses will follow almost anything 80 percent of the
time. High confidence, lift of 1, no information.

In [ ]:
all_rules = association_rules(fi, metric='confidence', min_threshold=0.0)
all_rules = all_rules[all_rules.consequents.apply(len) == 1]

high_conf = all_rules[all_rules.confidence > 0.80]
flat      = high_conf[(high_conf.lift > 0.97) & (high_conf.lift < 1.03)]

print(f'rules with confidence above 0.80 : {len(high_conf):,}')
print(f'   ... of which lift is about 1  : {len(flat):,}   <- confident and worthless')
print()
if len(flat):
    print('Examples:')
    for _, x in flat.head(4).iterrows():
        a = ' AND '.join(sorted(x.antecedents)); b = list(x.consequents)[0]
        print(f'   {a}  ->  {b}')
        print(f'      confidence {x.confidence:.2f}, but lift {x.lift:.2f}')
print()
print('Always sort by lift. Never by confidence alone.')

---
# Section 4: FP-Growth

Different route, same destination.

- **Apriori** generates candidates level by level and scans the data once per level.
- **FP-Growth** compresses the data into a tree and mines it. No candidates at all.

In [ ]:
t0 = time.perf_counter(); fi_ap = apriori(B, min_support=MIN_SUPPORT, use_colnames=True)
t_ap = time.perf_counter() - t0

t0 = time.perf_counter(); fi_fp = fpgrowth(B, min_support=MIN_SUPPORT, use_colnames=True)
t_fp = time.perf_counter() - t0

print(f'Apriori   : {len(fi_ap):,} itemsets in {t_ap:.3f}s')
print(f'FP-Growth : {len(fi_fp):,} itemsets in {t_fp:.3f}s')
print()
print(f'Same itemsets? {set(fi_ap.itemsets) == set(fi_fp.itemsets)}')

merged = (fi_ap.assign(k=fi_ap.itemsets.apply(frozenset))
          .merge(fi_fp.assign(k=fi_fp.itemsets.apply(frozenset)),
                 on='k', suffixes=('_ap', '_fp')))
print(f'Same supports? {np.allclose(merged.support_ap, merged.support_fp)}')
print()
print('Identical answers. Only the route differs.')

In [ ]:
print('TIMING, AT SEVERAL THRESHOLDS')
print()
print(f"{'min support':>13}{'itemsets':>11}{'apriori':>11}{'fpgrowth':>11}{'faster':>12}")
for ms in [0.30, 0.20, 0.10, 0.05, 0.02]:
    t0 = time.perf_counter(); a = apriori(B, min_support=ms);  ta = time.perf_counter() - t0
    t0 = time.perf_counter(); f = fpgrowth(B, min_support=ms); tf = time.perf_counter() - t0
    win = 'apriori' if ta < tf else 'fpgrowth'
    print(f'{ms:>13}{len(a):>11,}{ta:>10.4f}s{tf:>10.4f}s{win:>12}')
print()
print('SURPRISED? Apriori probably won.')

### Why FP-Growth loses on this dataset

Everything you read says FP-Growth is faster. On Ames it is not, and the reason
matters more than the result.

**FP-Growth pays an upfront cost.** It builds a tree before mining anything. That
cost is worth paying only when the alternative, generating and counting candidates,
is expensive.

**Here the alternative is cheap.** We have 9 facts. The largest possible candidate
set is tiny, and Apriori finishes before FP-Growth has finished building its tree.

**FP-Growth wins when:**

- There are many items, think thousands of products rather than 9 facts
- The support threshold is low, so the itemsets get long
- Transactions share long prefixes, so the tree compresses well

**A supermarket has 50,000 products.** That is where the tree earns its keep.
With 9 facts it is like hiring a removal van to carry one box.

The cell below builds progressively wider problems and shows exactly where the
answer flips.

**The honest lesson:** benchmark on your own data. The faster algorithm in the
textbook is not always the faster algorithm in your notebook.

In [ ]:
# Build a problem that looks like a supermarket: many items, and groups of
# things that genuinely travel together, so long itemsets form.
rng = np.random.default_rng(0)

def shop_like(n_rows, n_items, n_groups, group_size, p=0.25):
    M = np.zeros((n_rows, n_items), bool)
    groups = [rng.choice(n_items, group_size, replace=False) for _ in range(n_groups)]
    for r in range(n_rows):
        for g in groups:
            if rng.random() < p:
                M[r, g] = True
    return pd.DataFrame(M, columns=[f'item{i:03d}' for i in range(n_items)])

print(f"{'rows':>7}{'items':>7}{'itemsets':>11}{'apriori':>11}{'fpgrowth':>11}{'faster':>11}")
for rows, items, ng, gs, ms in [(3000, 20, 4, 5, 0.05),
                                (3000, 50, 6, 5, 0.05),
                                (3000, 80, 8, 6, 0.05)]:
    D = shop_like(rows, items, ng, gs)
    t0 = time.perf_counter(); a = apriori(D, min_support=ms);  ta = time.perf_counter() - t0
    t0 = time.perf_counter(); f = fpgrowth(D, min_support=ms); tf = time.perf_counter() - t0
    win = 'apriori' if ta < tf else 'FP-GROWTH'
    print(f'{rows:>7}{items:>7}{len(a):>11,}{ta:>10.2f}s{tf:>10.2f}s{win:>11}')
print()
print('Watch the itemsets column. That is what decides it.')
print('Few itemsets, Apriori wins. Tens of thousands, FP-Growth wins by a lot.')
print()
print('Ames gave us a few hundred itemsets. A supermarket gives millions.')

---
# Section 5: Which Features Travel Together

In [ ]:
print('THE STRONGEST RULE FOR EACH CONCLUSION')
print()
best = (rules.sort_values('lift', ascending=False)
             .drop_duplicates(subset='THEN'))
for _, r in best.iterrows():
    print(f'   IF    {r.IF}')
    print(f'   THEN  {r.THEN}')
    print(f'         support {r.support:.3f}   confidence {r.confidence:.2f}   '
          f'lift {r.lift:.2f}')
    print()

In [ ]:
# Which features avoid each other?
avoid = association_rules(fi, metric='lift', min_threshold=0)
avoid = avoid[(avoid.consequents.apply(len) == 1) & (avoid.lift < 0.9)]

if len(avoid):
    avoid['IF']   = avoid.antecedents.apply(lambda s: ' AND '.join(sorted(s)))
    avoid['THEN'] = avoid.consequents.apply(lambda s: list(s)[0])
    print('FEATURES THAT REPEL EACH OTHER   (lift below 0.9)')
    print()
    for _, r in avoid.sort_values('lift').drop_duplicates(subset='THEN').head(6).iterrows():
        print(f'   IF    {r.IF}')
        print(f'   THEN  rarely {r.THEN}   confidence {r.confidence:.2f}   lift {r.lift:.2f}')
        print()
    print('Lift below 1 means RARER than chance. These tell you what builders')
    print('do not put together.')
else:
    print('No rules with lift below 0.9 at this threshold.')
    print('Try lowering MIN_SUPPORT, or widening the lift cutoff.')

In [ ]:
plt.figure(figsize=(9.5, 6))
sc = plt.scatter(rules.support, rules.confidence, c=rules.lift,
                 s=30, cmap='viridis', alpha=.7)
plt.colorbar(sc, label='lift')
plt.xlabel('support'); plt.ylabel('confidence')
plt.title('Every rule, coloured by lift')
plt.tight_layout(); plt.show()

print('Bright points are the interesting rules.')
print('Notice that high confidence alone, high on the y axis, does not mean bright.')

---
# Section 6: Checking a New Record

A record arrives claiming a set of features. Do the rules find it plausible?

In [ ]:
def check(facts, top=6):
    """What do the rules say a house like this should also have?"""
    have = set(facts)
    unknown = have - set(B.columns)
    if unknown:
        print(f'Unknown facts: {unknown}'); return

    hits = rules[rules.antecedents.apply(lambda s: s <= have)]
    hits = hits[~hits.THEN.isin(have)]

    print('CLAIMED: ' + ', '.join(sorted(have)))
    print()
    if len(hits) == 0:
        print('   No rule fires. This combination is unusual in our data.')
        return
    hits = (hits.sort_values('confidence', ascending=False)
                .drop_duplicates(subset='THEN').head(top))
    print(f"   {'the rules expect':<24}{'conf':>7}{'lift':>7}")
    for _, r in hits.iterrows():
        flag = '   <- very likely' if r.confidence > 0.9 else ''
        print(f'   {r.THEN:<24}{r.confidence:>7.2f}{r.lift:>7.2f}{flag}')
    print()
    print('   Each line is a feature the record does NOT claim, but houses')
    print('   like this usually have. Anything above 0.90 is worth checking.')

check(['Large house', 'High quality', 'Built after 1990'])

In [ ]:
print('=' * 58)
check(['Large house', 'High quality', 'Many rooms', '2+ bathrooms'])
print()
print('=' * 58)
check(['Large lot'])

### What this is actually for

- **Data quality.** A record claiming a combination the rules say is near impossible
  deserves a second look.
- **Filling gaps.** If a field is missing, the rules tell you what is likely.
- **Understanding the housing stock.** Features cluster because of how houses get built,
  not by accident.